In [ ]:
# pip install transformers
# pip install torch
# pip install fuzzywuzzy

In [27]:
import pandas as pd
from gensim import corpora
from gensim.models import LdaModel


import torch
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification, AutoTokenizer, AutoModelForSequenceClassification
from torch.nn.functional import softmax

In [28]:
# Parquet-Datei laden
df = pd.read_parquet("all_parties_text_combined_prep.parquet")

In [29]:
# Testen mit den Parteitexten 
documents = df["Text_lemmatized"].apply(lambda x: x.split()).tolist()

# Wörterbuch erstellen (Mapping von Wörtern zu IDs)
dictionary = corpora.Dictionary(documents)

# Dokumente in Bag-of-Words (BoW) umwandeln
corpus = [dictionary.doc2bow(doc) for doc in documents]

# Wörterbuchgröße ausgeben
print(f"Wörterbuchgröße: {len(dictionary)} eindeutige Begriffe")

Wörterbuchgröße: 13506 eindeutige Begriffe


In [30]:
MODEL_NAME = "bert-base-german-cased"
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)  # 3 Labels für die Antwortmöglichkeiten


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-german-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
display(df_inputs)

,Party,Text,Question,Prediction,Antwort
0,Partei A,Thema 1: Wir sollten mehr Geld für Bildung aus...,Sollte mehr Geld für Bildung ausgegeben werden?,2,neutral
1,Partei B,Thema 2: Steuern sollten gesenkt werden.,Sollte mehr Geld für Bildung ausgegeben werden?,2,neutral
2,Partei A,Thema 1: Wir sollten mehr Geld für Bildung aus...,Sollten die Steuern gesenkt werden?,2,neutral
3,Partei B,Thema 2: Steuern sollten gesenkt werden.,Sollten die Steuern gesenkt werden?,2,neutral


In [96]:
# Dateien laden
parties_file = "all_parties_text_combined_prep.parquet"
questions_file = "wahlomaten_thesen_positionen.csv"

df_parties = pd.read_parquet(parties_file)
df_questions = pd.read_csv(questions_file)

In [97]:
# Umbennennen der Spalten zu den richtigen Parteinamen 
replace_dict = {
    'spd'   :   'SPD', 
    'cdu'   :   'CDU / CSU', 
    'gruene':   'GRÜNE', 
    'afd'   :   'AfD',
    'fdp'   :   'FDP', 
    'linke' :   'Die Linke'
}

df_parties["Party"] = df_parties["Party"].str.lower().str.strip().replace(replace_dict)
df_questions.rename(columns=replace_dict, inplace=True)

print("Spalten in df_parties:", df_parties["Party"].unique())
print("Spalten in df_questions:", df_questions.columns.tolist())

Spalten in df_parties: ['SPD' 'CDU / CSU' 'GRÜNE' 'AfD' 'FDP' 'Die Linke']
Spalten in df_questions: ['These', 'CDU / CSU', 'GRÜNE', 'SPD', 'AfD', 'Die Linke', 'FDP']


In [98]:
# Fehlende Werte in Leere Strings umwandeln 
df_parties = df_parties.fillna("")
df_questions = df_questions.fillna("")

In [99]:
# Filtern um Laufzeit zu kürzen 
df_parties = df_parties[df_parties["Party"] == "SPD"]
df_questions = df_questions[df_questions["These"] == "Alle Beschäftigten sollen bereits nach 40 Beitragsjahren ohne Abschläge in Rente gehen können."]

In [33]:
# verwenden eines vortrainiertes Modells 
MODEL_NAME = "oliverguhr/german-sentiment-bert"  # Trainiertes Modell für Sentiment-Analyse
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [82]:
from fuzzywuzzy import process

# Funktion zum Finden der ähnlichsten These
def match_question(these, question_list):
    match, score = process.extractOne(these, question_list)
    return match if score > 80 else None  # Nur Übereinstimmungen > 80% behalten

# Liste aller Fragen im Wahlomat
question_list = df_questions["These"].dropna().tolist()

# Fuzzy Matching durchführen
df_parties["Matched_These"] = df_parties["Text_no_stopwords"].apply(lambda x: match_question(x, question_list))

# Nur die übereinstimmenden Zeilen behalten
df_combined = df_parties.merge(df_questions, left_on="Matched_These", right_on="These", how="inner")


In [83]:
df_combined

,Party,Chapter,These_x,Text,Text_no_stopwords,These_no_stopwords,Text_lemmatized,These_lemmatized,Matched_These,These_y,CDU / CSU,GRÜNE,SPD,AfD,Die Linke,FDP
0,AfD,,Entwicklungspolitik,Die AfD fordert einen grundsätzlichen Strategi...,afd fordert grundsätzlichen strategiewechsel e...,entwicklungspolitik,afd fordern grundsätzlich Strategiewechsel Ent...,Entwicklungspolitik,Alle Beschäftigten sollen bereits nach 40 Beit...,Alle Beschäftigten sollen bereits nach 40 Beit...,stimme nicht zu,stimme nicht zu,stimme nicht zu,stimme nicht zu,stimme zu,stimme nicht zu
1,AfD,,Autonomie der Hochschulen stärken: Freiheit vo...,Deutschland muss ein Land der Spitzenforschung...,deutschland land spitzenforschung bleiben höhe...,autonomie hochschulen stärken freiheit forschu...,Deutschland Land Spitzenforschung bleiben hoch...,Autonomie Hochschule stärken Freiheit Forschun...,Alle Beschäftigten sollen bereits nach 40 Beit...,Alle Beschäftigten sollen bereits nach 40 Beit...,stimme nicht zu,stimme nicht zu,stimme nicht zu,stimme nicht zu,stimme zu,stimme nicht zu


In [84]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Beispiel-Thesen
these_1 = "Sollten die Steuern gesenkt werden?"
these_2 = "Steuersenkung: Sollten sie durchgeführt werden?"

# Vektorisierung (Bag of Words)
vectorizer = CountVectorizer().fit_transform([these_1, these_2])
vectors = vectorizer.toarray()

# Jaccard Similarity berechnen
similarity_score = cosine_similarity(vectors)[0][1]
print(f"Jaccard Similarity Score: {similarity_score}")


Jaccard Similarity Score: 0.39999999999999997


In [104]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# **3️⃣ Liste aller Wahlomat-Thesen**
question_list = df_questions["These"].tolist()

# **4️⃣ Jaccard Similarity Funktion**
def jaccard_similarity(text1, text2):
    words_text1 = set(text1.lower().split())
    words_text2 = set(text2.lower().split())
    intersection = words_text1.intersection(words_text2)
    union = words_text1.union(words_text2)
    return len(intersection) / len(union) if len(union) > 0 else 0

# **5️⃣ Finden der besten Übereinstimmung für jede Partei-These**
best_matches = []

for _, row in df_parties.iterrows():
    party = row["Party"]
    party_these = row["Text_lemmatized"]

    # Jaccard Similarity für jede Wahlomat-These berechnen
    similarities = [(question, jaccard_similarity(party_these, question)) for question in question_list]

    # Beste Übereinstimmung finden (höchste Jaccard-Score)
    best_match = max(similarities, key=lambda x: x[1]) if similarities else ("", 0)

    best_matches.append({
        "Party": party,
        "Party_These": party_these,
        "Best_Matching_Question": best_match[0],
        "Jaccard_Score": best_match[1]
    })

# **6️⃣ Ergebnisse als DataFrame speichern**
df_best_matches = pd.DataFrame(best_matches)

# **7️⃣ Speichern der Ergebnisse**
df_best_matches.to_csv("jaccard_matching_results.csv", index=False)


In [105]:
df_best_matches_sorted = df_best_matches.sort_values(by="Jaccard_Score", ascending=False)
display(df_best_matches_sorted.head(50))
df_best_matches_sorted.to_csv("jaccard_matching_results.csv", index=False)

,Party,Party_These,Best_Matching_Question,Jaccard_Score
19,SPD,Einwanderungsgesellschaft brauchen Bildungssys...,Alle Beschäftigten sollen bereits nach 40 Beit...,0.029412
32,SPD,gut Absicherung alt kernversprech Sozialstaat ...,Alle Beschäftigten sollen bereits nach 40 Beit...,0.023810
10,SPD,versprechen SPD kämpfen Arbeitsplatz gut Arbei...,Alle Beschäftigten sollen bereits nach 40 Beit...,0.020408
84,SPD,stellen klar Seenotrettung Verpflichtung inter...,Alle Beschäftigten sollen bereits nach 40 Beit...,0.019417
31,SPD,preis Lebensmittel seit empfndlich steigen dav...,Alle Beschäftigten sollen bereits nach 40 Beit...,0.019048
37,SPD,Kind Anfang gut Chance gut aufwachsen sichern ...,Alle Beschäftigten sollen bereits nach 40 Beit...,0.018519
86,SPD,Frieden Freiheit selbstverständlich müssen era...,Alle Beschäftigten sollen bereits nach 40 Beit...,0.018182
53,SPD,Staat Arbeitgeber attraktiv Fächendeckendem Ho...,Alle Beschäftigten sollen bereits nach 40 Beit...,0.018182
72,SPD,Schutz Diskriminierung Grundgesetz gg explizit...,Alle Beschäftigten sollen bereits nach 40 Beit...,0.017544
33,SPD,gesetzlich Rentenversicherung erster stark Säu...,Alle Beschäftigten sollen bereits nach 40 Beit...,0.017143


In [24]:
# **5. Batchweise Tokenisierung und Verarbeitung**
batch_size = 16  # Verarbeite 16 Samples auf einmal
predictions = []

for i in range(0, len(df_model_input), batch_size):
    batch = df_model_input.iloc[i : i + batch_size]

    encoded_inputs = tokenizer(
        list(batch["Text"]),
        padding=True,
        truncation=True,
        max_length=512,  # Beschränkung auf 512 Tokens
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**encoded_inputs)

    logits = outputs.logits
    probs = softmax(logits, dim=1)
    batch_predictions = torch.argmax(probs, dim=1).tolist()  # 0 = negativ, 1 = neutral, 2 = positiv
    predictions.extend(batch_predictions)

# **6. Vorhersagen speichern**
df_model_input["Prediction"] = predictions
label_mapping = {0: "stimme nicht zu", 1: "neutral", 2: "stimme zu"}
df_model_input["Antwort"] = df_model_input["Prediction"].map(label_mapping)

# **7. Ergebnisse speichern**
output_file = "bert_multiple_choice_results.csv"
df_model_input.to_csv(output_file, index=False)
print(f"✅ Ergebnisse gespeichert in {output_file}")

✅ Ergebnisse gespeichert in bert_multiple_choice_results.csv


In [25]:
df_model_input

,Party,Text,Question,Prediction,Antwort
0,AfD,Frage: Alle Beschäftigten sollen bereits nach ...,Alle Beschäftigten sollen bereits nach 40 Beit...,2,stimme zu
1,AfD,Frage: Alle Beschäftigten sollen bereits nach ...,Alle Beschäftigten sollen bereits nach 40 Beit...,2,stimme zu
2,AfD,Frage: Alle Beschäftigten sollen bereits nach ...,Alle Beschäftigten sollen bereits nach 40 Beit...,2,stimme zu
3,AfD,Frage: Alle Beschäftigten sollen bereits nach ...,Alle Beschäftigten sollen bereits nach 40 Beit...,2,stimme zu
4,AfD,Frage: Alle Beschäftigten sollen bereits nach ...,Alle Beschäftigten sollen bereits nach 40 Beit...,2,stimme zu
...,...,...,...,...,...
3111,AfD,Frage: Ökologische Landwirtschaft soll stärker...,Ökologische Landwirtschaft soll stärker geförd...,2,stimme zu
3112,AfD,Frage: Ökologische Landwirtschaft soll stärker...,Ökologische Landwirtschaft soll stärker geförd...,2,stimme zu
3113,AfD,Frage: Ökologische Landwirtschaft soll stärker...,Ökologische Landwirtschaft soll stärker geförd...,2,stimme zu
3114,AfD,Frage: Ökologische Landwirtschaft soll stärker...,Ökologische Landwirtschaft soll stärker geförd...,2,stimme zu


In [26]:
import pandas as pd

# BERT-Vorhersagen umbenennen, um Verwechslungen zu vermeiden
df_model_input = df_model_input.rename(columns={"Antwort": "BERT_Antwort"})

# Wahlomat-Daten umstrukturieren:
#    - Spalten "CDU/CSU", "SPD", "AfD" etc. in eine Spalte "Party" mit Werten "stimme zu", "stimme nicht zu", "neutral"
df_actual = df_questions.melt(id_vars=["These"], var_name="Party", value_name="Wahlomat_Antwort")

# Mergen der BERT-Vorhersagen mit den tatsächlichen Wahlomat-Antworten
df_comparison = df_model_input.merge(df_actual, left_on=["Question", "Party"], right_on=["These", "Party"], how="left")

# Unnötige Spalte "These" entfernen (weil sie mit "Question" identisch ist)
df_comparison = df_comparison.drop(columns=["These"])

# Neue Spalte hinzufügen: Vergleich der Antworten
df_comparison["Übereinstimmung"] = df_comparison["BERT_Antwort"] == df_comparison["Wahlomat_Antwort"]

df_comparison.head(38)


,Party,Text,Question,Prediction,BERT_Antwort,Wahlomat_Antwort,Übereinstimmung
0,AfD,Frage: Alle Beschäftigten sollen bereits nach ...,Alle Beschäftigten sollen bereits nach 40 Beit...,2,stimme zu,stimme nicht zu,False
1,AfD,Frage: Alle Beschäftigten sollen bereits nach ...,Alle Beschäftigten sollen bereits nach 40 Beit...,2,stimme zu,stimme nicht zu,False
2,AfD,Frage: Alle Beschäftigten sollen bereits nach ...,Alle Beschäftigten sollen bereits nach 40 Beit...,2,stimme zu,stimme nicht zu,False
3,AfD,Frage: Alle Beschäftigten sollen bereits nach ...,Alle Beschäftigten sollen bereits nach 40 Beit...,2,stimme zu,stimme nicht zu,False
4,AfD,Frage: Alle Beschäftigten sollen bereits nach ...,Alle Beschäftigten sollen bereits nach 40 Beit...,2,stimme zu,stimme nicht zu,False
5,AfD,Frage: Alle Beschäftigten sollen bereits nach ...,Alle Beschäftigten sollen bereits nach 40 Beit...,0,stimme nicht zu,stimme nicht zu,True
6,AfD,Frage: Alle Beschäftigten sollen bereits nach ...,Alle Beschäftigten sollen bereits nach 40 Beit...,2,stimme zu,stimme nicht zu,False
7,AfD,Frage: Alle Beschäftigten sollen bereits nach ...,Alle Beschäftigten sollen bereits nach 40 Beit...,2,stimme zu,stimme nicht zu,False
8,AfD,Frage: Alle Beschäftigten sollen bereits nach ...,Alle Beschäftigten sollen bereits nach 40 Beit...,2,stimme zu,stimme nicht zu,False
9,AfD,Frage: Alle Beschäftigten sollen bereits nach ...,Alle Beschäftigten sollen bereits nach 40 Beit...,2,stimme zu,stimme nicht zu,False
